In [ ]:
import subprocess
import sys
from datetime import datetime, timedelta
import os

# Install required packages for MODIS data downloading
packages = ['requests', 'rasterio', 'xarray']
for package in packages:
    try:
        __import__(package)
        print(f"{package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"{package} installed successfully")

import requests
import pandas as pd
import numpy as np

# MOD13A2 is MODIS Vegetation Indices (NDVI/EVI) with 16-day composite
# Date range: December 1, 2000 to May 24, 2016
start_date = datetime(2000, 12, 1)
end_date = datetime(2016, 5, 24)

print(f"MOD13A2 Download Configuration")
print(f"=" * 60)
print(f"Dataset: MOD13A2 (MODIS Vegetation Indices)")
print(f"Date Range: {start_date.date()} to {end_date.date()}")
print(f"Duration: {(end_date - start_date).days} days")

# MODIS tiles covering Uttar Pradesh, Haryana, and Punjab (IGP region)
# The sinusoidal grid tiles for northern India:
tiles = ['h25v05', 'h25v06', 'h26v05', 'h26v06']

print(f"\nMODIS Tiles (covering UP, Haryana, Punjab):")
for tile in tiles:
    print(f"  - {tile}")

# Create data directories
data_dir = "/media/sam/writable/Sam Rice Yield Pred/data"
mod13a2_dir = os.path.join(data_dir, "MOD13A2")
os.makedirs(mod13a2_dir, exist_ok=True)

print(f"\nData Directory: {mod13a2_dir}")
print(f"\n" + "=" * 60)
print("✓ Configuration ready")
print("✓ Ready to proceed with NASA LAADS DAAC downloads")

In [ ]:
import getpass
import json

# Prompt for NASA Earthdata bearer token (used for download authentication)
print("=" * 60)
print("NASA Earthdata Authentication")
print("=" * 60)
bearer_token = getpass.getpass("Enter your NASA Earthdata bearer token: ")

# Headers for downloading (CMR search does not require auth)
headers = {
    "Authorization": f"Bearer {bearer_token}"
}

print(f"\n✓ Token stored securely")
print(f"✓ Ready to query NASA CMR for MOD13A2")

# Use NASA's Common Metadata Repository (CMR) Search API
# CMR is the canonical metadata service for all NASA Earth science data
print(f"\n" + "=" * 60)
print(f"Querying MOD13A2 granules from NASA CMR")
print("=" * 60)

cmr_url = "https://cmr.earthdata.nasa.gov/search/granules.json"
collection_short_name = "MOD13A2"
collection_version = "061"  # Collection 6.1

# Convert dates to ISO 8601 format for CMR
start_str = start_date.strftime('%Y-%m-%dT%H:%M:%SZ')
end_str = end_date.strftime('%Y-%m-%dT23:59:59Z')

# Query CMR for each tile by filtering producer_granule_id (contains tile name)
search_results = {}
total_files = 0

for tile in tiles:
    print(f"\nQuerying tile {tile}...")
    
    all_granules = []
    page_num = 1
    
    while True:
        params = {
            "short_name": collection_short_name,
            "version": collection_version,
            "temporal": "{},{}".format(start_str, end_str),
            "producer_granule_id": "*{}*".format(tile),  # Wildcard match on tile name
            "options[producer_granule_id][pattern]": "true",
            "page_size": 2000,
            "page_num": page_num
        }
        
        try:
            response = requests.get(cmr_url, params=params, timeout=60)
            response.raise_for_status()
            data = response.json()
            
            granules = data.get('feed', {}).get('entry', [])
            if not granules:
                break
            
            all_granules.extend(granules)
            
            # Check if more pages exist
            if len(granules) < 2000:
                break
            page_num += 1
            
        except requests.exceptions.RequestException as e:
            print("  Error querying CMR: {}".format(str(e)))
            break
    
    # Extract download URLs from granules
    granule_info = []
    for g in all_granules:
        title = g.get('title', '')
        # Find the download link (HDF file)
        for link in g.get('links', []):
            href = link.get('href', '')
            if href.endswith('.hdf') and 'http' in href:
                granule_info.append({
                    'filename': href.split('/')[-1],
                    'download_url': href,
                    'title': title
                })
                break
    
    search_results[tile] = {
        'count': len(granule_info),
        'files': granule_info
    }
    total_files += len(granule_info)
    print("  Found {} files".format(len(granule_info)))

print(f"\n" + "=" * 60)
print(f"Summary: Found {total_files} total MOD13A2 granules")
print(f"Tiles: {len(tiles)}")
print(f"Date range: {start_str} to {end_str}")
print("=" * 60)

# Show a sample to verify
if total_files > 0:
    sample_tile = next(t for t in tiles if search_results[t]['count'] > 0)
    sample_file = search_results[sample_tile]['files'][0]
    print(f"\nSample file from tile {sample_tile}:")
    print(f"  Filename: {sample_file['filename']}")
    print(f"  URL: {sample_file['download_url']}")

print(f"\n✓ Ready to proceed with downloading {total_files} files")

In [ ]:
import os

# Download MOD13A2 files for each tile
# Note: NASA Earthdata authentication uses redirects; allow them with the session
print("=" * 60)
print("Downloading MOD13A2 Files from NASA")
print("=" * 60)

downloaded_count = 0
failed_count = 0
skipped_count = 0

# Use a session that handles redirects properly while preserving the auth header
session = requests.Session()
session.headers.update(headers)

for tile, tile_info in search_results.items():
    files = tile_info.get('files', [])
    num_files = len(files)
    
    print("\n[Tile {}] Processing {} files...".format(tile, num_files))
    
    # Create subdirectory for each tile
    tile_dir = os.path.join(mod13a2_dir, tile)
    os.makedirs(tile_dir, exist_ok=True)
    
    for idx, file_info in enumerate(files, 1):
        filename = file_info['filename']
        download_url = file_info['download_url']
        filepath = os.path.join(tile_dir, filename)
        
        # Skip if already downloaded
        if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
            skipped_count += 1
            continue
        
        try:
            # Stream the download
            response = session.get(download_url, timeout=120, stream=True, allow_redirects=True)
            response.raise_for_status()
            
            # Write to file
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            
            file_size_mb = os.path.getsize(filepath) / (1024 * 1024)
            print("  [{}/{}] {} ({:.1f} MB) ✓".format(idx, num_files, filename, file_size_mb))
            downloaded_count += 1
            
        except Exception as e:
            print("  [{}/{}] {} - FAILED: {}".format(idx, num_files, filename, str(e)))
            failed_count += 1
            # Clean up partial download
            if os.path.exists(filepath):
                os.remove(filepath)
            continue

print("\n" + "=" * 60)
print("Download Summary")
print("=" * 60)
print("✓ Successfully downloaded: {} files".format(downloaded_count))
print("✗ Failed: {} files".format(failed_count))
print("⊘ Skipped (already exist): {} files".format(skipped_count))
print("Total processed: {}".format(downloaded_count + failed_count + skipped_count))
print("\nData location: {}".format(mod13a2_dir))
print("=" * 60)

# Verify downloaded files
print("\nVerifying downloaded files...")
for tile in tiles:
    tile_dir = os.path.join(mod13a2_dir, tile)
    if os.path.exists(tile_dir):
        files = [f for f in os.listdir(tile_dir) if f.endswith('.hdf')]
        total_size_gb = sum(os.path.getsize(os.path.join(tile_dir, f)) for f in files) / (1024**3)
        print("  {}: {} HDF files ({:.2f} GB)".format(tile, len(files), total_size_gb))

In [ ]:
# ============================================================
# CSIF Download Configuration
# ============================================================
# CSIF (Contiguous Solar-Induced Chlorophyll Fluorescence) - Zhang et al.
# Hosted at OSF: https://osf.io/8xqy6/
#
# Two product variants needed:
#   1. clear.inst v2 (CSIF_v2/ folder)  -> for Dec 2000 only (all-sky starts 2001)
#   2. all.daily v1  (all-sky/ folder)  -> for 2001-01-01 onward
#
# Both are 4-day composites at 0.05 deg spatial resolution.

import os
from datetime import datetime, timedelta

# OSF project node
OSF_NODE_ID = "8xqy6"
OSF_API_BASE = "https://api.osf.io/v2"

# Folder IDs (discovered via OSF API exploration)
CSIF_FOLDERS = {
    "clear_inst_v2": {
        "name": "CSIF_v2",
        "folder_id": "5c9b7619aae20b0017b090c9",
        "filename_pattern": "OCO2.SIF.clear.inst.{year}{doy:03d}.v2.nc",
    },
    "all_daily": {
        "name": "all-sky",
        "folder_id": "5bd9a1961385910017689662",
        "filename_pattern": "OCO2.SIF.all.daily.{year}{doy:03d}.nc",
    },
}

# Date range (same as MOD13A2)
csif_start_date = datetime(2000, 12, 1)
csif_end_date = datetime(2016, 5, 24)

# Bounding box: UP, Haryana, Punjab + 2 deg buffer on all sides
bbox = {
    "lat_min": 22.0,
    "lat_max": 34.5,
    "lon_min": 72.0,
    "lon_max": 86.5,
}

# Storage directories
csif_dir = "/media/sam/writable/Sam Rice Yield Pred/data/CSIF"
csif_clear_dir = os.path.join(csif_dir, "clear_inst_v2_subset")  # Dec 2000 only
csif_all_dir = os.path.join(csif_dir, "all_daily_subset")        # 2001+
os.makedirs(csif_clear_dir, exist_ok=True)
os.makedirs(csif_all_dir, exist_ok=True)

print("CSIF Download Configuration")
print("=" * 60)
print("Date Range : {} -> {}".format(csif_start_date.date(), csif_end_date.date()))
print("Bounding box (lat,lon): ({}, {}) -> ({}, {})".format(
    bbox["lat_min"], bbox["lon_min"], bbox["lat_max"], bbox["lon_max"]))
print()
print("Hybrid strategy:")
print("  Dec 2000      -> CSIF_v2 (clear.inst v2)")
print("  2001-01-01 -> 2016-05-24 -> all-sky (all.daily v1)")
print()
print("Storage:")
print("  clear.inst v2 subset: {}".format(csif_clear_dir))
print("  all.daily subset    : {}".format(csif_all_dir))


def doy_from_date(dt):
    """Return Julian day-of-year (1-indexed) for a datetime."""
    return (dt - datetime(dt.year, 1, 1)).days + 1


def expected_doys_for_year(year, start_dt, end_dt):
    """
    CSIF 4-day composites use DOY = 1, 5, 9, ... within each year.
    Return the list of DOYs in the given year that fall within
    [start_dt, end_dt].
    """
    doys = []
    for doy in range(1, 366, 4):
        try:
            file_date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        except ValueError:
            continue
        if start_dt <= file_date <= end_dt:
            doys.append(doy)
    return doys


print("\nExpected file count (estimate):")
print("  Dec 2000 (clear.inst v2): {} files".format(
    len(expected_doys_for_year(2000, csif_start_date, csif_end_date))))
all_daily_total = 0
for y in range(2001, 2017):
    all_daily_total += len(expected_doys_for_year(y, csif_start_date, csif_end_date))
print("  2001-2016 (all.daily)   : {} files".format(all_daily_total))


In [ ]:
# ============================================================
# Enumerate CSIF files via OSF API
# ============================================================
# OSF organizes files as: project -> folder -> year subfolder -> NC files
# We must walk the tree to collect file IDs and download URLs.

def osf_list_folder(folder_url):
    """List entries (files/folders) under an OSF storage folder URL, with pagination."""
    entries = []
    next_url = folder_url
    while next_url:
        resp = requests.get(next_url, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        entries.extend(data.get("data", []))
        next_url = data.get("links", {}).get("next")
    return entries


def osf_folder_url(folder_id):
    return "{}/nodes/{}/files/osfstorage/{}/".format(
        OSF_API_BASE, OSF_NODE_ID, folder_id
    )


# Build inventory of files we want to download
csif_inventory = {"clear_inst_v2": [], "all_daily": []}

print("Enumerating CSIF files from OSF...")
print("=" * 60)

for variant_key, variant_info in CSIF_FOLDERS.items():
    print("\nVariant: {} ({})".format(variant_key, variant_info["name"]))
    print("-" * 60)

    # List year subfolders inside this variant's folder
    year_folders = osf_list_folder(osf_folder_url(variant_info["folder_id"]))

    for yf in year_folders:
        if yf["attributes"]["kind"] != "folder":
            continue
        year_name = yf["attributes"]["name"]
        try:
            year = int(year_name)
        except ValueError:
            continue

        # Hybrid strategy filter:
        #   clear_inst_v2 -> ONLY 2000 (Dec 2000)
        #   all_daily     -> 2001 through 2016
        if variant_key == "clear_inst_v2" and year != 2000:
            continue
        if variant_key == "all_daily" and not (2001 <= year <= 2016):
            continue

        # Compute the DOYs we need from this year
        wanted_doys = set(expected_doys_for_year(year, csif_start_date, csif_end_date))
        if not wanted_doys:
            continue

        # List files in this year folder
        year_folder_id = yf["attributes"]["path"].strip("/")
        year_files = osf_list_folder(osf_folder_url(year_folder_id))

        matched = 0
        for ff in year_files:
            if ff["attributes"]["kind"] != "file":
                continue
            fname = ff["attributes"]["name"]
            if not fname.endswith(".nc"):
                continue

            # Parse DOY out of filename: OCO2.SIF.{sky}.{temp}.YYYYDDD[.v2].nc
            m = re.search(r"\.(\d{4})(\d{3})(?:\.v\d+)?\.nc$", fname)
            if not m:
                continue
            file_year = int(m.group(1))
            file_doy = int(m.group(2))
            if file_year != year or file_doy not in wanted_doys:
                continue

            download_url = ff["links"]["download"]
            csif_inventory[variant_key].append({
                "year": file_year,
                "doy": file_doy,
                "filename": fname,
                "download_url": download_url,
            })
            matched += 1

        print("  {}: {} files matched (wanted {})".format(year, matched, len(wanted_doys)))

print("\n" + "=" * 60)
print("Inventory Summary")
print("=" * 60)
for k, v in csif_inventory.items():
    print("  {}: {} files".format(k, len(v)))
total_csif = sum(len(v) for v in csif_inventory.values())
print("  TOTAL: {} files".format(total_csif))

# Show a sample
if csif_inventory["clear_inst_v2"]:
    s = csif_inventory["clear_inst_v2"][0]
    print("\nSample (clear_inst_v2): {} -> {}".format(s["filename"], s["download_url"]))
if csif_inventory["all_daily"]:
    s = csif_inventory["all_daily"][0]
    print("Sample (all_daily)    : {} -> {}".format(s["filename"], s["download_url"]))


In [ ]:
# ============================================================
# Download CSIF files and save spatial subset (UP/Haryana/Punjab + 2deg buffer)
# ============================================================
# Strategy: stream each global NetCDF to a temp file, open with xarray,
# crop to bbox, write subset, then delete the global file.

import tempfile
import xarray as xr

print("Downloading CSIF files and subsetting to bounding box...")
print("=" * 60)
print("bbox: lat [{}, {}], lon [{}, {}]".format(
    bbox["lat_min"], bbox["lat_max"], bbox["lon_min"], bbox["lon_max"]))
print()

csif_session = requests.Session()


def download_and_subset(file_info, output_dir):
    """Download a CSIF NetCDF, write a bbox-cropped subset, return (success, msg)."""
    out_path = os.path.join(output_dir, file_info["filename"])

    # Skip if already subset
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return ("skip", "exists")

    # Download to temp file
    tmp_fd, tmp_path = tempfile.mkstemp(suffix=".nc")
    os.close(tmp_fd)
    try:
        resp = csif_session.get(file_info["download_url"], stream=True, timeout=180)
        resp.raise_for_status()
        with open(tmp_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1 << 16):
                if chunk:
                    f.write(chunk)

        # Open, subset, write
        ds = xr.open_dataset(tmp_path)

        # CSIF NetCDF uses 'lat' and 'lon' coords; latitudes typically descend (90 -> -90)
        lat_name = "lat" if "lat" in ds.coords else "latitude"
        lon_name = "lon" if "lon" in ds.coords else "longitude"

        # Determine ordering of lat axis
        lat_vals = ds[lat_name].values
        if lat_vals[0] > lat_vals[-1]:
            lat_slice = slice(bbox["lat_max"], bbox["lat_min"])
        else:
            lat_slice = slice(bbox["lat_min"], bbox["lat_max"])
        lon_slice = slice(bbox["lon_min"], bbox["lon_max"])

        ds_sub = ds.sel({lat_name: lat_slice, lon_name: lon_slice})

        # Write with NetCDF4 + compression
        encoding = {var: {"zlib": True, "complevel": 4}
                    for var in ds_sub.data_vars}
        ds_sub.to_netcdf(out_path, encoding=encoding)
        ds.close()
        ds_sub.close()
        return ("ok", "{} KB".format(os.path.getsize(out_path) // 1024))
    except Exception as e:
        if os.path.exists(out_path):
            os.remove(out_path)
        return ("fail", str(e))
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


# Run downloads for both variants
variant_to_dir = {
    "clear_inst_v2": csif_clear_dir,
    "all_daily": csif_all_dir,
}

totals = {"ok": 0, "skip": 0, "fail": 0}
for variant_key, files in csif_inventory.items():
    out_dir = variant_to_dir[variant_key]
    print("\n[{}] {} files -> {}".format(variant_key, len(files), out_dir))
    for i, file_info in enumerate(files, 1):
        status, msg = download_and_subset(file_info, out_dir)
        totals[status] += 1
        if status == "fail":
            print("  [{}/{}] {} FAILED: {}".format(i, len(files), file_info["filename"], msg))
        elif i % 25 == 0 or i == len(files):
            print("  [{}/{}] processed ({} ok, {} skip, {} fail so far)".format(
                i, len(files), totals["ok"], totals["skip"], totals["fail"]))

print("\n" + "=" * 60)
print("CSIF Download + Subset Summary")
print("=" * 60)
print("  Downloaded + subset : {}".format(totals["ok"]))
print("  Skipped (existed)   : {}".format(totals["skip"]))
print("  Failed              : {}".format(totals["fail"]))

# Final size check
print("\nFinal subset storage:")
for variant_key, d in variant_to_dir.items():
    files = [f for f in os.listdir(d) if f.endswith(".nc")]
    total_mb = sum(os.path.getsize(os.path.join(d, f)) for f in files) / (1024 * 1024)
    print("  {}: {} files ({:.2f} MB)".format(variant_key, len(files), total_mb))


In [ ]:
# ============================================================
# CHIRPS Download Configuration
# ============================================================
# CHIRPS v2.0 daily precipitation at 0.05 deg resolution
# Hosted by UCSB Climate Hazards Center:
#   https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_daily/netcdf/p05/
#
# Files are annual: chirps-v2.0.YYYY.days_p05.nc (~1.2 GB each)
# We download, spatially subset (same bbox as CSIF), then DELETE the full file.

CHIRPS_BASE_URL = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_daily/netcdf/p05"

# Same date range and bbox as CSIF / MOD13A2
chirps_start_date = datetime(2000, 12, 1)
chirps_end_date = datetime(2016, 5, 24)

# Years we need (full annual files - we slice by date during 16-day aggregation)
chirps_years = list(range(chirps_start_date.year, chirps_end_date.year + 1))

# Storage
chirps_dir = "/media/sam/writable/Sam Rice Yield Pred/data/CHIRPS"
chirps_subset_dir = os.path.join(chirps_dir, "daily_subset")        # daily, bbox-cropped, by year
chirps_16day_dir = os.path.join(chirps_dir, "16day_aggregated")     # 16-day sums aligned to MOD13A2
os.makedirs(chirps_subset_dir, exist_ok=True)
os.makedirs(chirps_16day_dir, exist_ok=True)

print("CHIRPS Download Configuration")
print("=" * 60)
print("Source           : {}".format(CHIRPS_BASE_URL))
print("Date range       : {} -> {}".format(chirps_start_date.date(), chirps_end_date.date()))
print("Years            : {} ({} files)".format(chirps_years[0], len(chirps_years)) +
      " -> {}".format(chirps_years[-1]))
print("Spatial bbox     : lat [{}, {}], lon [{}, {}]".format(
    bbox["lat_min"], bbox["lat_max"], bbox["lon_min"], bbox["lon_max"]))
print()
print("Storage:")
print("  daily subsets (by year) : {}".format(chirps_subset_dir))
print("  16-day aggregated       : {}".format(chirps_16day_dir))


In [ ]:
# ============================================================
# Download CHIRPS annual files, spatially subset, delete full file
# ============================================================
# Each annual file is ~1.2 GB. We stream to tempfile, crop, save subset,
# then delete the tempfile. Per-year subset is ~6-10 MB after compression.

import tempfile
import xarray as xr

chirps_session = requests.Session()

def fetch_and_subset_chirps_year(year):
    """Download CHIRPS annual file for a year, subset to bbox, return (status, msg)."""
    fname = "chirps-v2.0.{}.days_p05.nc".format(year)
    url = "{}/{}".format(CHIRPS_BASE_URL, fname)
    out_path = os.path.join(chirps_subset_dir, fname)

    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return ("skip", "exists")

    tmp_fd, tmp_path = tempfile.mkstemp(suffix=".nc")
    os.close(tmp_fd)
    try:
        resp = chirps_session.get(url, stream=True, timeout=600)
        resp.raise_for_status()
        total_bytes = int(resp.headers.get("Content-Length", 0))
        downloaded = 0
        with open(tmp_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)

        # Open and subset
        ds = xr.open_dataset(tmp_path)
        lat_name = "latitude" if "latitude" in ds.coords else "lat"
        lon_name = "longitude" if "longitude" in ds.coords else "lon"

        lat_vals = ds[lat_name].values
        if lat_vals[0] > lat_vals[-1]:
            lat_slice = slice(bbox["lat_max"], bbox["lat_min"])
        else:
            lat_slice = slice(bbox["lat_min"], bbox["lat_max"])

        ds_sub = ds.sel({lat_name: lat_slice, lon_name: slice(bbox["lon_min"], bbox["lon_max"])})

        # Write subset with compression
        encoding = {var: {"zlib": True, "complevel": 4} for var in ds_sub.data_vars}
        ds_sub.to_netcdf(out_path, encoding=encoding)
        ds.close()
        ds_sub.close()
        size_mb = os.path.getsize(out_path) / (1024 * 1024)
        return ("ok", "{:.1f} MB".format(size_mb))
    except Exception as e:
        if os.path.exists(out_path):
            os.remove(out_path)
        return ("fail", str(e))
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


print("Downloading + subsetting CHIRPS annual files...")
print("=" * 60)
chirps_totals = {"ok": 0, "skip": 0, "fail": 0}
for year in chirps_years:
    print("  [{}] downloading...".format(year), end=" ", flush=True)
    status, msg = fetch_and_subset_chirps_year(year)
    chirps_totals[status] += 1
    print("{} ({})".format(status, msg))

print("\n" + "=" * 60)
print("CHIRPS Download Summary")
print("=" * 60)
print("  Subset OK     : {}".format(chirps_totals["ok"]))
print("  Skipped       : {}".format(chirps_totals["skip"]))
print("  Failed        : {}".format(chirps_totals["fail"]))

# Storage report
total_size_mb = 0
for f in sorted(os.listdir(chirps_subset_dir)):
    if f.endswith(".nc"):
        fp = os.path.join(chirps_subset_dir, f)
        sz = os.path.getsize(fp) / (1024 * 1024)
        total_size_mb += sz
        print("  {}: {:.1f} MB".format(f, sz))
print("\n  TOTAL: {:.1f} MB across {} files".format(
    total_size_mb, len([f for f in os.listdir(chirps_subset_dir) if f.endswith(".nc")])))


In [ ]:
# ============================================================
# Aggregate CHIRPS daily -> 16-day SUMS aligned to MOD13A2 windows
# ============================================================
# MOD13A2 16-day composites start at DOY 1, 17, 33, ..., 353 within each year.
# For each such window covering our date range, SUM the daily precipitation.
# Sums are appropriate for precip (additive, total rainfall in window).
#
# Note: We iterate over annual files one at a time with xr.open_dataset()
# (no xr.open_mfdataset) to avoid pulling in dask on Python 3.6.

import numpy as np


def mod13a2_16day_starts(start_dt, end_dt):
    """Return list of (year, doy, window_start, window_end) tuples.

    Each year independently uses DOY = 1, 17, 33, ..., 353 (23 windows/year).
    Windows are 16 days each, except the last in each year (353 -> end of year).
    """
    starts = []
    for year in range(start_dt.year, end_dt.year + 1):
        for doy in range(1, 366, 16):
            try:
                window_start = datetime(year, 1, 1) + timedelta(days=doy - 1)
            except ValueError:
                continue
            year_end = datetime(year, 12, 31)
            window_end = min(window_start + timedelta(days=15), year_end)
            if window_end < start_dt or window_start > end_dt:
                continue
            starts.append((year, doy, window_start, window_end))
    return starts


windows = mod13a2_16day_starts(chirps_start_date, chirps_end_date)
print("Generated {} MOD13A2-aligned 16-day windows".format(len(windows)))
print("  First: {} (year {} DOY {:03d}) -> {}".format(
    windows[0][2].date(), windows[0][0], windows[0][1], windows[0][3].date()))
print("  Last : {} (year {} DOY {:03d}) -> {}".format(
    windows[-1][2].date(), windows[-1][0], windows[-1][1], windows[-1][3].date()))

# Group windows by year so we only need one annual file open at a time
windows_by_year = {}
for w in windows:
    windows_by_year.setdefault(w[0], []).append(w)

# Process each annual CHIRPS file independently
print("\nAggregating per-year...")
agg_arrays = []
agg_starts = []
agg_year_doy = []
ref_lat = None
ref_lon = None
lat_name = None
lon_name = None

for year in sorted(windows_by_year.keys()):
    nc_path = os.path.join(chirps_subset_dir,
                           "chirps-v2.0.{}.days_p05.nc".format(year))
    if not os.path.exists(nc_path):
        print("  [{}] MISSING file, skipping".format(year))
        continue

    ds = xr.open_dataset(nc_path)

    if lat_name is None:
        lat_name = "latitude" if "latitude" in ds.coords else "lat"
        lon_name = "longitude" if "longitude" in ds.coords else "lon"
        ref_lat = ds[lat_name]
        ref_lon = ds[lon_name]

    precip_var = "precip" if "precip" in ds.data_vars else list(ds.data_vars)[0]

    year_windows = windows_by_year[year]
    year_count = 0
    for (y, doy, w_start, w_end) in year_windows:
        sel = ds[precip_var].sel(time=slice(np.datetime64(w_start), np.datetime64(w_end)))
        if sel.time.size == 0:
            continue
        agg_arrays.append(sel.sum(dim="time", skipna=True).values)
        agg_starts.append(np.datetime64(w_start))
        agg_year_doy.append((y, doy))
        year_count += 1

    print("  [{}] {} windows aggregated (file: {})".format(
        year, year_count, os.path.basename(nc_path)))
    ds.close()

agg_stack = np.stack(agg_arrays, axis=0)
print("\nAggregated array shape: {}".format(agg_stack.shape))

# Build output Dataset
out_ds = xr.Dataset(
    {
        "precip_16day_sum": (
            ("time", lat_name, lon_name),
            agg_stack,
            {"units": "mm", "long_name": "16-day total precipitation aligned to MOD13A2"},
        ),
    },
    coords={
        "time": agg_starts,
        lat_name: ref_lat,
        lon_name: ref_lon,
    },
)
out_ds["year"] = ("time", [yd[0] for yd in agg_year_doy])
out_ds["doy"] = ("time", [yd[1] for yd in agg_year_doy])
out_ds.attrs["description"] = (
    "CHIRPS v2.0 0.05 deg precipitation summed over MOD13A2 16-day composite "
    "windows (DOY = 1, 17, 33, ..., 353 within each calendar year). "
    "Spatially subset to UP/Haryana/Punjab + 2 deg buffer."
)

agg_path = os.path.join(chirps_16day_dir, "chirps_16day_2000-2016.nc")
encoding = {"precip_16day_sum": {"zlib": True, "complevel": 4}}
out_ds.to_netcdf(agg_path, encoding=encoding)
print("\nWrote aggregated file: {}".format(agg_path))
print("  Size: {:.1f} MB".format(os.path.getsize(agg_path) / (1024 * 1024)))
print("  Windows: {}".format(out_ds.dims["time"]))
out_ds.close()

print("\n✓ CHIRPS 16-day aggregation complete")


In [ ]:
# ============================================================
# CRUNCEP V7 Solar Radiation (FSDS): 6-hourly -> 16-day means
# ============================================================
# Inputs: 204 monthly NetCDF files at /media/sam/writable/Sam Rice Yield Pred/Solr
#   File pattern: clmforc.cruncep.V7.c2016.0.5d.Solr.YYYY-MM.nc
#   Variable    : FSDS (W/m^2, 6-hourly instantaneous)
#   Grid        : 0.5 deg, lat ascending (-89.75..89.75), lon 0-360 (0.25..359.75)
#
# Standard protocol for rice yield:
#   step 1: 6-hourly W/m^2 -> daily mean W/m^2
#   step 2: daily mean -> mean over each MOD13A2 16-day window
#
# Output: ONE aggregated NetCDF aligned to MOD13A2 windows (DOY=1,17,...,353)
# Then we can delete the 25 GB Solr/ source directory.

import glob

cruncep_src_dir = "/media/sam/writable/Sam Rice Yield Pred/Solr"
cruncep_out_dir = "/media/sam/writable/Sam Rice Yield Pred/data/CRUNCEP"
os.makedirs(cruncep_out_dir, exist_ok=True)

cruncep_start_date = datetime(2000, 12, 1)
cruncep_end_date = datetime(2016, 5, 24)

# Reuse bbox from earlier cells (already in 0-360 compatible range)
print("CRUNCEP Solar Radiation Processing")
print("=" * 60)
print("Source dir : {}".format(cruncep_src_dir))
print("Output dir : {}".format(cruncep_out_dir))
print("Date range : {} -> {}".format(cruncep_start_date.date(), cruncep_end_date.date()))
print("bbox       : lat [{}, {}], lon [{}, {}]".format(
    bbox["lat_min"], bbox["lat_max"], bbox["lon_min"], bbox["lon_max"]))

src_files = sorted(glob.glob(os.path.join(cruncep_src_dir, "clmforc.cruncep.V7.c2016.0.5d.Solr.*.nc")))
print("\nFound {} monthly source files".format(len(src_files)))
print("  First: {}".format(os.path.basename(src_files[0])))
print("  Last : {}".format(os.path.basename(src_files[-1])))


In [ ]:
# ============================================================
# Spatial subset + daily-mean accumulation across all months
# ============================================================
# For each monthly file:
#   - open, slice to bbox spatially
#   - resample 6-hourly -> daily mean (4 readings per day)
#   - append daily slices to growing in-memory arrays
# Final: one (n_days, n_lat, n_lon) array covering all 204 months.
# Subset size per timestep is tiny (~26x30 = 780 cells), so this fits in memory easily.

import re as _re

daily_means = []      # list of (n_days_in_month, n_lat, n_lon) arrays
daily_dates = []      # list of np.datetime64 (one per day)
ref_lat = None
ref_lon = None

print("Subsetting + daily-averaging each monthly file...")
print("=" * 60)
for i, fpath in enumerate(src_files, 1):
    fname = os.path.basename(fpath)
    m = _re.search(r"\.(\d{4})-(\d{2})\.nc$", fname)
    if not m:
        print("  Skipping (unrecognized name): {}".format(fname))
        continue

    ds = xr.open_dataset(fpath)
    # FSDS is the variable we want
    fsds = ds["FSDS"]

    # Spatial subset (lat ascending, lon 0-360 — bbox is in same convention for India)
    fsds_sub = fsds.sel(
        lat=slice(bbox["lat_min"], bbox["lat_max"]),
        lon=slice(bbox["lon_min"], bbox["lon_max"]),
    )

    # Capture reference lat/lon once
    if ref_lat is None:
        ref_lat = fsds_sub["lat"].values.copy()
        ref_lon = fsds_sub["lon"].values.copy()

    # Resample 6-hourly -> daily mean.
    # The time coord is cftime/object; convert to a pandas DatetimeIndex via pd.to_datetime
    # for reliable resampling.
    times = pd.to_datetime([str(t) for t in fsds_sub["time"].values])
    fsds_sub = fsds_sub.assign_coords(time=("time", times))
    daily = fsds_sub.resample(time="1D").mean()

    daily_means.append(daily.values)
    daily_dates.extend([np.datetime64(d) for d in pd.to_datetime(daily["time"].values).to_pydatetime()])

    if i % 24 == 0 or i == len(src_files):
        print("  [{}/{}] processed {}".format(i, len(src_files), fname))
    ds.close()

# Stack into single array
daily_array = np.concatenate(daily_means, axis=0)
daily_dates_arr = np.array(daily_dates)
print("\nStacked daily means shape: {}".format(daily_array.shape))
print("Days covered: {} -> {}".format(daily_dates_arr[0], daily_dates_arr[-1]))
print("Total daily timesteps: {}".format(len(daily_dates_arr)))

# Sanity: 2000-12-01 -> 2016-05-24 is 5654 days, but we have all of 2000-01..2016-12
expected_days = (datetime(2017, 1, 1) - datetime(2000, 1, 1)).days  # 6210
print("Expected days (2000-2016 inclusive): {}".format(expected_days))


In [ ]:
# ============================================================
# Aggregate to MOD13A2 16-day windows and write final NetCDF
# ============================================================
# Reuse the same window definition used for CHIRPS so all environmental
# datasets share an identical time axis.

cruncep_windows = mod13a2_16day_starts(cruncep_start_date, cruncep_end_date)
print("MOD13A2-aligned 16-day windows for CRUNCEP: {}".format(len(cruncep_windows)))

# Build an index from date -> position in daily_array for fast lookup
date_to_idx = {d: i for i, d in enumerate(daily_dates_arr)}

agg_values = []
agg_starts = []
agg_year_doy = []

for (year, doy, w_start, w_end) in cruncep_windows:
    # Collect daily indices in [w_start, w_end] inclusive
    days_in_window = []
    cur = w_start
    while cur <= w_end:
        idx = date_to_idx.get(np.datetime64(cur))
        if idx is not None:
            days_in_window.append(idx)
        cur = cur + timedelta(days=1)
    if not days_in_window:
        continue
    window_mean = np.nanmean(daily_array[days_in_window, :, :], axis=0)
    agg_values.append(window_mean)
    agg_starts.append(np.datetime64(w_start))
    agg_year_doy.append((year, doy))

agg_stack = np.stack(agg_values, axis=0)
print("Aggregated window array shape: {}".format(agg_stack.shape))

# Build output Dataset
out_ds = xr.Dataset(
    {
        "solr_16day_mean": (
            ("time", "lat", "lon"),
            agg_stack.astype("float32"),
            {
                "units": "W/m^2",
                "long_name": "Mean incident solar radiation over MOD13A2 16-day window",
                "source_variable": "FSDS",
            },
        ),
    },
    coords={
        "time": agg_starts,
        "lat": ref_lat,
        "lon": ref_lon,
    },
)
out_ds["year"] = ("time", [yd[0] for yd in agg_year_doy])
out_ds["doy"] = ("time", [yd[1] for yd in agg_year_doy])
out_ds.attrs["description"] = (
    "CRUNCEP V7 incident solar radiation (FSDS) averaged over MOD13A2 16-day "
    "composite windows (DOY = 1, 17, ..., 353 within each calendar year). "
    "Workflow: 6-hourly W/m^2 -> daily mean -> 16-day mean. "
    "Spatially subset to UP/Haryana/Punjab + 2 deg buffer (lat 22-34.5, lon 72-86.5). "
    "Native resolution 0.5 deg; longitude is 0-360 convention (unchanged from source)."
)

out_path = os.path.join(cruncep_out_dir, "cruncep_solr_16day_2000-2016.nc")
encoding = {"solr_16day_mean": {"zlib": True, "complevel": 4}}
out_ds.to_netcdf(out_path, encoding=encoding)
print("\nWrote: {}".format(out_path))
print("  Size: {:.2f} MB".format(os.path.getsize(out_path) / (1024 * 1024)))
print("  Windows: {}".format(out_ds.dims["time"]))
out_ds.close()

print("\n✓ CRUNCEP 16-day aggregation complete")


In [ ]:
# ============================================================
# CRUNCEP cleanup: verify aggregated file, then delete 25 GB source dir
# ============================================================
# Safety: only delete if the aggregated file exists AND has expected windows.

import shutil

agg_path = os.path.join(cruncep_out_dir, "cruncep_solr_16day_2000-2016.nc")

if not os.path.exists(agg_path):
    print("Aggregated file not found. Refusing to delete source.")
else:
    verify = xr.open_dataset(agg_path)
    n_windows = verify.dims["time"]
    verify.close()

    expected_min_windows = 350   # ~23/yr * 15 yrs + partial = 350+
    if n_windows < expected_min_windows:
        print("WARNING: aggregated file has only {} windows (expected >= {}).".format(
            n_windows, expected_min_windows))
        print("Refusing to delete source for safety.")
    else:
        # Show what we're about to delete
        n_files = len([f for f in os.listdir(cruncep_src_dir) if f.endswith('.nc')])
        size_gb = sum(os.path.getsize(os.path.join(cruncep_src_dir, f))
                      for f in os.listdir(cruncep_src_dir)) / (1024 ** 3)
        print("About to delete:")
        print("  Directory: {}".format(cruncep_src_dir))
        print("  Files    : {}".format(n_files))
        print("  Size     : {:.2f} GB".format(size_gb))
        print()
        print("Aggregated output retained at: {}".format(agg_path))
        print("  Size: {:.2f} MB, windows: {}".format(
            os.path.getsize(agg_path) / (1024 * 1024), n_windows))
        print()
        # Delete
        shutil.rmtree(cruncep_src_dir)
        print("Deleted {}".format(cruncep_src_dir))
        print("\n✓ {:.2f} GB freed".format(size_gb))
